# M3 微調(RF-DETR)— Colab

用 EPFL 多視角多類別資料微調 RF-DETR，驗證能不能偵測刀具/砧板/食材等小物件。

**先做**:上方選單 `執行階段 → 變更執行階段類型 → GPU(T4)`。

**要上傳的檔案**:`data/m3_finetune_mv/dataset.zip`(在你電腦專案裡，~16MB）。

⚠ EPFL 為 CC-NC，此為驗證用、模型不出貨。

In [ ]:
# 1) 檢查 GPU + 安裝套件（train,loggers 才含訓練依賴 pytorch_lightning）
!nvidia-smi -L
!pip -q install "rfdetr[train,loggers]" supervision

In [ ]:
# 2) 上傳 dataset.zip 並解壓（跳出選檔時選 data/m3_finetune_mv/dataset.zip）
from google.colab import files
up = files.upload()
!unzip -q -o dataset.zip -d /content/m3ds
!ls /content/m3ds   # 應看到 train valid test

In [ ]:
# 3) （對照）微調前：COCO 預訓練模型在 test 上偵測到什麼（只會有 COCO 類別，抓不到砧板/食材）
import glob, os
from PIL import Image
from rfdetr import RFDETRNano
base = RFDETRNano()
for p in sorted(glob.glob('/content/m3ds/test/*.jpg'))[:5]:
    det = base.predict(Image.open(p).convert('RGB'), threshold=0.3)
    names = det.data.get('class_name') if det.data else None
    labs = [str(names[i]) for i in range(len(det))] if names is not None else []
    print(os.path.basename(p), '->', labs)

In [ ]:
# 4) 微調 nano（重點）。過程會印每個 epoch 的 validation mAP → 看刀具/砧板/食材 AP 有沒有往上
from rfdetr import RFDETRNano
model = RFDETRNano()
model.train(
    dataset_dir='/content/m3ds',
    epochs=60,
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    resolution=704,
    output_dir='/content/out_nano',
)
# OOM 的話：batch_size=2、resolution=640

In [ ]:
# 5) 驗證：載入微調權重（不 optimize！）+ 用自訂中文類別預測
from rfdetr import RFDETRNano   # medium 換 RFDETRMedium、out_nano 換 out_medium
model = RFDETRNano(pretrain_weights="/content/out_nano/checkpoint_best_regular.pth", num_classes=11)

import glob, os
from PIL import Image
NAMES={0:"人",1:"刀具",2:"砧板",3:"食材",4:"鍋鏟",5:"鍋子",6:"手",7:"容器",8:"抹布",9:"夾子",10:"手套"}
for p in sorted(glob.glob("/content/m3ds/test/*.jpg"))[:8]:
    det = model.predict(Image.open(p).convert("RGB"), threshold=0.3)
    print(os.path.basename(p), "->", [NAMES.get(int(c),int(c)) for c in det.class_id])

In [ ]:
# 6) 視覺化一張並下載
import supervision as sv, numpy as np, cv2, glob
from PIL import Image
from google.colab import files
p = sorted(glob.glob('/content/m3ds/test/*.jpg'))[0]
img = Image.open(p).convert('RGB')
det = model.predict(img, threshold=0.3)
vis = sv.BoxAnnotator().annotate(np.array(img).copy(), det)
vis = sv.LabelAnnotator().annotate(vis, det)
cv2.imwrite('/content/pred.jpg', cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
files.download('/content/pred.jpg')

In [ ]:
# 7) 下載微調權重（.pth 一定可下；ONNX 匯出常因版本卡，真實部署再處理）
from google.colab import files
files.download("/content/out_nano/checkpoint_best_regular.pth")
# 若要 ONNX：!pip install --force-reinstall onnx==1.16.2 onnxscript → 重啟 → model.export(output_dir="/content/onnx")

## 之後:也跑 medium 比較

把第 4 格的 `RFDETRNano` 換成 `RFDETRMedium`、`output_dir='/content/out_medium'`，重跑第 4~7 格，**比較 nano vs medium 微調後的 knife/砧板 AP**，再決定用哪個變體（變體尚未鎖定）。

把訓練過程印出的 val mAP、和微調後 test 偵測結果貼回給 Claude，一起判讀。